## TASK 4 — Multiple Regression (with 1–2 macro drivers)

In [1]:
# import libraries
import pandas as pd, yfinance as yf

# select assets and benchmark
TICKER = ["NVDA", "AMD", "INTC"]
BENCH  = ["QQQ"] 
TICKER = TICKER + BENCH

# 3 year period
end = pd.Timestamp.today().normalize()
start = end - pd.DateOffset(months=36)

# download adjusted close prices
df = yf.download(TICKER, start=start, end=end, interval="1d", auto_adjust=True)[["Close"]]

# flatten column names if multi-index
# i am not sure why the template appends an extra index on the columns
df.columns = [col if isinstance(col, str) else col[1] for col in df.columns]

# reset index to make Date a column
df = df.reset_index()

# sort values by date
df = df.sort_values("Date")

# forward fill and drop any NAs
df = df.ffill().dropna()

# compute returns
df_returns = df[TICKER].pct_change().rename(columns=lambda x: x+ "_Return")

# align return columns to match df
price_cols = df.columns[1:]
df_returns = df_returns[[col + "_Return" for col in price_cols]]

# concatting two datas
df = pd.concat([df, df_returns], axis=1)

# fill returns of the first row to 0
df.fillna(0, inplace=True)

# convert necessary variables to numeric
# consumes a lot of time despite values being numeric already
variables = TICKER + [t + "_Return" for t in TICKER]
for v in variables:
    df[v] = pd.to_numeric(df[v], errors='coerce')

# save files
df.to_csv("prices_clean.csv", index=False)

# ==============================
# TRANSFORMING QUANTITATIVE DATA
# ==============================

# compress wide-format returns into long format
long_df = df.melt(id_vars="Date", value_vars=[t + "_Return" for t in TICKER], var_name="Ticker", value_name="Return")

# clean ticker names
long_df["Ticker"] = long_df["Ticker"].str.replace("_Return", "")

# convert ticker to categorical variables
long_df["Ticker"] = long_df["Ticker"].astype("category")

# save files
long_df.to_csv("returns_long_format.csv", index=False)

[*********************100%***********************]  4 of 4 completed


In [2]:
# multiple_regression_models.py
import numpy as np
import statsmodels.api as sm
from datetime import timedelta

# load file
df = pd.read_csv("prices_clean.csv", parse_dates=["Date"]).sort_values("Date")

# build portofolio return
asset_returns = ["NVDA_Return", "AMD_Return", "INTC_Return"]
missing_assets = [c for c in asset_returns if c not in df.columns]
if missing_assets:
    raise KeyError(f"Required asset return columns missing in prices_clean.csv: {missing_assets}")

df["Portfolio_Return"] = df[asset_returns].mean(axis=1)


In [3]:
# force df columns into single-level before merging anything
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[1] if isinstance(c, tuple) else c for c in df.columns]

# IEF_Return check, download it if it doesn't exist
if "IEF_Return" not in df.columns:
    print("Downloading clean IEF data...")
    
    ief = yf.download(
        "IEF",
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        progress=False
    )
    
    # --- FIX ALL POSSIBLE COLUMN/INDEX ISSUES --- #
    
    # 1. Flatten MultiIndex if needed
    if isinstance(ief.columns, pd.MultiIndex):
        ief.columns = ['_'.join([str(c) for c in col if c]) for col in ief.columns]
    
    # 2. Ensure index has the correct name BEFORE reset
    ief.index.name = "Date"
    
    # 3. Reset index → produce a Date column
    ief = ief.reset_index()
    
    # 4. Now force the Close column to exist
    close_candidates = [c for c in ief.columns if "Close" in c]
    
    if len(close_candidates) == 0:
        raise KeyError(f"No Close column in IEF after cleaning. Columns={ief.columns}")
    
    close_col = close_candidates[0]
    
    # 5. Rename Close → IEF
    ief = ief.rename(columns={close_col: "IEF"})
    
    # 6. Compute returns
    ief["IEF_Return"] = ief["IEF"].pct_change().fillna(0)
    
    # 7. Merge
    df = df.merge(ief[["Date", "IEF_Return"]], on="Date", how="left")
    df["IEF_Return"] = df["IEF_Return"].ffill().bfill()
    
    print("IEF_Return successfully merged.")

# feature engineering (lags and vol)
# portfolio asset
df = df.sort_values("Date").reset_index(drop=True)
df["Return"] = df["Portfolio_Return"]  # response
df["Lag1"] = df["Return"].shift(1)
df["Lag2"] = df["Return"].shift(2)
df["Vol20"] = df["Return"].rolling(window=20, min_periods=10).std()  # volatility proxy (20-day)

# momentum
df["Mom20"] = df["Portfolio_Return"].rolling(20).apply(lambda x: (x+1).prod() - 1)

# drop rows with NaNs used in regressions
d = df.dropna(subset=["Return","Lag1","Lag2"]).copy()

IEF_Return successfully merged.


In [4]:
# fit model function
def fit_model(df_, cols):
    X = sm.add_constant(df_[cols])
    y = df_["Return"]
    model = sm.OLS(y, X).fit()
    return model

# M1: lag-only
m1_cols = ["Lag1", "Lag2"]

# M2: lag + benchmark (QQQ_Return)
m2_cols = ["Lag1", "Lag2", "QQQ_Return"]

# M3: lag + benchmark + macro (IEF_Return)
m3_cols = ["Lag1", "Lag2", "QQQ_Return", "IEF_Return"]

models = {}
models["M1 (Lag1,Lag2)"] = fit_model(d, m1_cols)
models["M2 (+QQQ)"] = fit_model(d, m2_cols)
models["M3 (+QQQ,+IEF)"] = fit_model(d, m3_cols)

# comparison table (Adj-R2 / AIC / BIC)
cmp = []
for name, mod in models.items():
    cmp.append({
        "Model": name,
        "Adj_R2": mod.rsquared_adj,
        "AIC": mod.aic,
        "BIC": mod.bic,
        "N": int(mod.nobs)
    })
cmp_df = pd.DataFrame(cmp).set_index("Model").round(6)
print("\nModel comparison (higher Adj_R2 better; lower AIC/BIC better):\n")
print(cmp_df)


Model comparison (higher Adj_R2 better; lower AIC/BIC better):

                  Adj_R2          AIC          BIC    N
Model                                                  
M1 (Lag1,Lag2)  0.001941 -3369.128487 -3355.268267  750
M2 (+QQQ)       0.671294 -4201.120426 -4182.640133  750
M3 (+QQQ,+IEF)  0.672276 -4202.368965 -4179.268599  750


In [5]:
# short summaries for chosen model candidates
print("\nSummary (top lines) for each model:\n")
for name, mod in models.items():
    print("-----", name, "-----")
    print(mod.summary().tables[0])
    print(mod.summary().tables[1])   # coefficients table
    print("\n")


Summary (top lines) for each model:

----- M1 (Lag1,Lag2) -----
                            OLS Regression Results                            
Dep. Variable:                 Return   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.728
Date:                Sun, 16 Nov 2025   Prob (F-statistic):              0.178
Time:                        20:50:31   Log-Likelihood:                 1687.6
No. Observations:                 750   AIC:                            -3369.
Df Residuals:                     747   BIC:                            -3355.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------